In [1]:
import numpy as np
import random
import gc
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

# Load data
X_train = np.load("D:/PomegranateGuard/dataset/X_train.npy")
X_val   = np.load("D:/PomegranateGuard/dataset/X_val.npy")

y_train = np.load("D:/PomegranateGuard/dataset/y_train.npy")
y_val   = np.load("D:/PomegranateGuard/dataset/y_val.npy")

X_train = X_train.astype('float16')
X_val = X_val.astype('float16')

X_train_small = X_train[:500]
y_train_small = y_train[:500]
X_val_small   = X_val[:200]
y_val_small   = y_val[:200]

D:\Anaconda\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
def build_model(learning_rate, dropout_rate, filters):
    model = Sequential([
        Conv2D(filters, (3,3), activation='relu', input_shape=(224,224,3)),
        MaxPooling2D(2,2),

        Conv2D(filters*2, (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Flatten(),
        Dense(64, activation='relu'),   # reduced from 128
        Dropout(dropout_rate),
        Dense(4, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [3]:
def fitness(solution):
    lr, dropout, filters = solution

    model = build_model(
        learning_rate=lr,
        dropout_rate=dropout,
        filters=int(filters)
    )

    history = model.fit(
        X_train_small, y_train_small,
        epochs=2,               # 🔹 only 1 epoch
        batch_size=8,           # 🔹 small batch
        validation_data=(X_val_small, y_val_small),
        verbose=0
    )

    val_acc = history.history['val_accuracy'][-1]

    # 🔹 MEMORY CLEANUP (VERY IMPORTANT)
    K.clear_session()
    del model
    gc.collect()

    return val_acc

In [4]:
def honey_badger_optimization(iterations=100):

    best_solution = None
    best_score = 0

    for i in range(iterations):

        solution = [
            random.uniform(0.0002, 0.0008),
            random.uniform(0.35, 0.55),
            random.choice([16, 32])  # 🔹 reduced filters
        ]

        score = fitness(solution)

        if score > best_score:
            best_score = score
            best_solution = solution

        if (i+1) % 10 == 0:
            print(f"Iteration {i+1} | Best Val Accuracy: {best_score:.4f}")

    return best_solution, best_score

In [5]:
best_params, best_val_acc = honey_badger_optimization()

print("\nBest Parameters Found:")
print("Learning Rate:", best_params[0])
print("Dropout Rate:", best_params[1])
print("Filters:", best_params[2])
print("Best Validation Accuracy:", best_val_acc)

D:\Anaconda\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Iteration 10 | Best Val Accuracy: 0.8700
Iteration 20 | Best Val Accuracy: 0.8700
Iteration 30 | Best Val Accuracy: 0.8900
Iteration 40 | Best Val Accuracy: 0.8900
Iteration 50 | Best Val Accuracy: 0.8900
Iteration 60 | Best Val Accuracy: 0.8900
Iteration 70 | Best Val Accuracy: 0.8900
Iteration 80 | Best Val Accuracy: 0.8900
Iteration 90 | Best Val Accuracy: 0.8900
Iteration 100 | Best Val Accuracy: 0.8900

Best Parameters Found:
Learning Rate: 0.0006146065009157053
Dropout Rate: 0.39183388404167396
Filters: 16
Best Validation Accuracy: 0.8899999856948853


In [6]:
with open("D:/PomegranateGuard/optimization/hbo_results.txt", "w") as f:
    f.write(f"Best Learning Rate: {best_params[0]}\n")
    f.write(f"Best Dropout Rate: {best_params[1]}\n")
    f.write(f"Best Filters: {best_params[2]}\n")
    f.write(f"Best Validation Accuracy: {best_val_acc}\n")

print("Results Saved Successfully")

Results Saved Successfully
